In [ ]:
import pandas as pd
from pathlib import Path
from pandasql import sqldf
import sqlite3
import numpy as np
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
import os

In [ ]:
db_path = Path(r'C:\Users\Art\Documents\civ5\Civ5DebugDatabase.db')

In [ ]:
cnx = sqlite3.connect(db_path)

In [ ]:
pd.read_sql_query('select * from features', cnx)

In [ ]:
CIV_TAG_TO_TEXT_MAP = {
    "CIVILIZATION_AMERICA": "America",
    "CIVILIZATION_ARABIA": "Arabia",
    "CIVILIZATION_AZTEC": "The Aztecs",
    "CIVILIZATION_CHINA": "China",
    "CIVILIZATION_EGYPT": "Egypt",
    "CIVILIZATION_ENGLAND": "England",
    "CIVILIZATION_FRANCE": "France",
    "CIVILIZATION_GERMANY": "Germany",
    "CIVILIZATION_GREECE": "Greece",
    "CIVILIZATION_INDIA": "India",
    "CIVILIZATION_IROQUOIS": "The Iroquois",
    "CIVILIZATION_JAPAN": "Japan",
    "CIVILIZATION_OTTOMAN": "The Ottomans",
    "CIVILIZATION_PERSIA": "Persia",
    "CIVILIZATION_ROME": "Rome",
    "CIVILIZATION_RUSSIA": "Russia",
    "CIVILIZATION_SIAM": "Siam",
    "CIVILIZATION_SONGHAI": "Songhai",
    "CIVILIZATION_MINOR": "City State",
    "CIVILIZATION_BARBARIAN": "Barbarians",
    "CIVILIZATION_MONGOL": "Mongolia",
    "CIVILIZATION_INCA": "The Inca",
    "CIVILIZATION_SPAIN": "Spain",
    "CIVILIZATION_POLYNESIA": "Polynesia",
    "CIVILIZATION_DENMARK": "Denmark",
    "CIVILIZATION_KOREA": "Korea",
    "CIVILIZATION_BABYLON": "Babylon",
    "CIVILIZATION_AUSTRIA": "Austria",
    "CIVILIZATION_BYZANTIUM": "Byzantium",
    "CIVILIZATION_CARTHAGE": "Carthage",
    "CIVILIZATION_CELTS": "The Celts",
    "CIVILIZATION_ETHIOPIA": "Ethiopia",
    "CIVILIZATION_HUNS": "The Huns",
    "CIVILIZATION_MAYA": "The Maya",
    "CIVILIZATION_NETHERLANDS": "The Netherlands",
    "CIVILIZATION_SWEDEN": "Sweden",
    "CIVILIZATION_ASSYRIA": "Assyria",
    "CIVILIZATION_BRAZIL": "Brazil",
    "CIVILIZATION_INDONESIA": "Indonesia",
    "CIVILIZATION_MOROCCO": "Morocco",
    "CIVILIZATION_POLAND": "Poland",
    "CIVILIZATION_PORTUGAL": "Portugal",
    "CIVILIZATION_SHOSHONE": "The Shoshone",
    "CIVILIZATION_VENICE": "Venice",
    "CIVILIZATION_ZULU": "The Zulus"
}

In [ ]:
civ_flavors_df = pd.read_sql_query('''
SELECT
    CivilizationType,
    FlavorType,
    Flavor
FROM Leader_Flavors, Civilization_Leaders
WHERE Leader_Flavors.LeaderType = Civilization_Leaders.LeaderheadType
AND CivilizationType != "CIVILIZATION_MINOR"
AND CivilizationType != "CIVILIZATION_BARBARIAN"

UNION

SELECT
    CivilizationType,
    MajorCivApproachType AS FlavorType,
    Bias AS Flavor
FROM Leader_MajorCivApproachBiases, Civilization_Leaders
WHERE Leader_MajorCivApproachBiases.LeaderType = Civilization_Leaders.LeaderheadType
AND CivilizationType != "CIVILIZATION_MINOR"
AND CivilizationType != "CIVILIZATION_BARBARIAN"
''', cnx)

civ_flavors_df['civ'] = civ_flavors_df['CivilizationType'].map(CIV_TAG_TO_TEXT_MAP)
civ_flavors_df = sqldf('''
SELECT
  civ,
  FlavorType,
  Flavor
FROM civ_flavors_df
GROUP BY civ, FlavorType
''')
civ_flavors_df

In [ ]:
leader_personality_df = pd.read_sql_query('''
SELECT
    Type,
    CivilizationType,
    Description,
    Personality,
    PrimaryVictoryPursuit,
    SecondaryVictoryPursuit
FROM Leaders, Civilization_Leaders
WHERE Leaders.Type = Civilization_Leaders.LeaderheadType
AND CivilizationType != "CIVILIZATION_MINOR"
AND CivilizationType != "CIVILIZATION_BARBARIAN"
''', cnx)

leader_personality_df['civ'] = leader_personality_df['CivilizationType'].map(CIV_TAG_TO_TEXT_MAP)
leader_personality_df['Personality'] = leader_personality_df['Personality'].str.replace('PERSONALITY_', '')
leader_personality_df['PrimaryVictoryPursuit'] = leader_personality_df['PrimaryVictoryPursuit'].str.replace('VICTORY_PURSUIT_', '')
leader_personality_df['SecondaryVictoryPursuit'] = leader_personality_df['SecondaryVictoryPursuit'].str.replace('VICTORY_PURSUIT_', '')
leader_personality_df

In [ ]:
pd.pivot_table(civ_flavors_df, index=['civ'], columns=['FlavorType'], values='Flavor').merge(
    leader_personality_df,
    on='civ',
    how='left'
).to_csv('civ_flavors.csv')

In [ ]:
# pd.pivot_table(civ_flavors_df, index=['civ'], columns=['FlavorType'], values='Flavor').to_csv('civ_flavors.csv')

In [ ]:
civ_colors_df = pd.read_sql_query('''
SELECT
    Civilizations.Type,
    Colors.Red * 255 AS red,
    Colors.Green * 255 AS green,
    Colors.Blue * 255 AS blue
FROM
    Civilizations, PlayerColors, Colors
WHERE
    Civilizations.DefaultPlayerColor = PlayerColors.Type AND
    PlayerColors.PrimaryColor = Colors.Type
''', cnx)


civ_colors_df['civ'] = civ_colors_df['Type'].map(CIV_TAG_TO_TEXT_MAP)
civ_colors_df = civ_colors_df.drop(columns=['Type']).round()
civ_colors_df

In [ ]:
civ_colors_df.to_csv('civ_colors.csv')

In [ ]:
civ_bg_colors_df = pd.read_sql_query('''
SELECT
    Civilizations.Type,
    Colors.Red * 255 AS red,
    Colors.Green * 255 AS green,
    Colors.Blue * 255 AS blue
FROM
    Civilizations, PlayerColors, Colors
WHERE
    Civilizations.DefaultPlayerColor = PlayerColors.Type AND
    PlayerColors.SecondaryColor = Colors.Type
''', cnx)


civ_bg_colors_df['civ'] = civ_bg_colors_df['Type'].map(CIV_TAG_TO_TEXT_MAP)
civ_bg_colors_df = civ_bg_colors_df.drop(columns=['Type'])
civ_bg_colors_df

In [ ]:
civ_bg_colors_df.to_csv('civ_bg_colors.csv')

In [ ]:
pd.read_sql_query("SELECT * FROM PlayerColors", cnx)

In [ ]:
pd.read_sql_query("SELECT * FROM Colors", cnx)

In [ ]:
pd.read_sql_query("SELECT * FROM Civilizations", cnx)

In [ ]:
pd.read_sql_query("SELECT * FROM Civilizations", cnx).columns

In [ ]:
pd.read_sql_query("SELECT * FROM Civilizations", cnx)[['DefaultPlayerColor']]